# Train a student — full 2M, KD + pruning (resumable)
Trains one architecture with per-epoch checkpoints (`last.pt` every epoch,
`best.pt` on improvement). **Multi-session:** to continue, attach *this
notebook's previous output* and the run resumes via `--resume`.

**Inputs to attach:** `skripsi-edgenmten-id` (valid.tsv + train.tsv), the
tokenizer Model, and — for `DATA_MODE='kd'` — the KD-gen output (`kd_train.tsv`).
GPU T4. **Save Version at the end** so `results/runs/...` persists.


In [ ]:
import os 
import sys
import glob
import shutil
import subprocess
import torch
import json as js

ARCH='gru'            # gru | lstm | transformer
DATA_MODE='baseline'  # baseline | kd   (kd needs kd_train.tsv attached)
QAT=False              # quantization aware training (fake quantization)
PRUNE=False            # gradual magnitude pruning during training
TARGET_SPARSITY=0.75  # final fraction of Linear weights zeroed

REPO='https://github.com/0wLzz/Edge-NMT.git'

if not os.path.isdir('Edge-NMT'):
    subprocess.run(['git','clone','--depth','1',REPO], check=True)

os.chdir('/kaggle/working/Edge-NMT'); sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd(), '| arch', ARCH, '| mode', DATA_MODE, '| prune', PRUNE)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q',
    'sentencepiece','pyyaml','tqdm'], check=True)

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY (slow)')

In [ ]:
# Assemble data/processed with the tsvs this run needs
os.makedirs('data/processed', exist_ok=True)

def link(src, name):
    dst='data/processed/'+name
    if os.path.islink(dst) or os.path.exists(dst): os.remove(dst)
    os.symlink(src, dst); print(name,'->',src)

link(glob.glob('/kaggle/input/**/valid.tsv', recursive=True)[0], 'valid.tsv')

if DATA_MODE=='kd':
    kd = glob.glob('/kaggle/input/**/kd_train.tsv', recursive=True)
    assert kd, 'DATA_MODE=kd but no kd_train.tsv attached — attach the KD-gen output'
    link(kd[0], 'kd_train.tsv')
else:
    link(glob.glob('/kaggle/input/**/train.tsv', recursive=True)[0], 'train.tsv')

TOKENIZER = glob.glob('/kaggle/input/**/spm_en_id.model', recursive=True)[0]
print('tokenizer:',TOKENIZER)

In [ ]:
# Resume: restore a prior run dir (this notebook's previous output) into results/runs/
os.makedirs('results/runs', exist_ok=True)

prior = [d for d in glob.glob('/kaggle/input/**/results/runs/*', recursive=True) if os.path.isdir(d)]

for d in prior:
    dst = 'results/runs/'+os.path.basename(d)
    if not os.path.exists(dst):
        shutil.copytree(d, dst); print('restored run:', dst)

if not prior: 
    print('no prior run attached — starting fresh')

In [ ]:
# Train. --resume continues the latest matching run (fresh if none). Same command
# every session. Checkpoints: results/runs/<ts>_<name>/{last.pt (per epoch), best.pt}.
cmd = [sys.executable,'-m',
     'model.training.train',
     '--arch',ARCH,
     '--data-mode',DATA_MODE,
     '--dataset-dir','data/processed',
     '--tokenizer-model',TOKENIZER,
     '--resume',
]

if QAT:
    cmd += ['--qat']

if PRUNE: 
     cmd += ['--prune','--target-sparsity',str(TARGET_SPARSITY)]

print('RUN', *cmd, flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# Show result. SAVE VERSION after this to persist the run dir.

runs = sorted(glob.glob('results/runs/*/history.json'))

if runs:
    h = js.load(open(runs[-1]))
    print('best_val_loss:', h.get('best_val_loss'), '| epochs:', len(h.get('epochs',[])))
    print('run dir:', os.path.dirname(runs[-1]))
else:
    print('no history yet — check the training output above')